*Aljoscha Rheinwalt <aljoscha.rheinwalt@uni-potsdam.de> (2026)*

---

# CUDA block matching of two Sentinel-2 rasters

This tutorial estimates a dense pixel displacement field from the 2018 image to the 2020 image with masked normalized cross-correlation (NCC), checks each match in the reverse direction, writes georeferenced GeoTIFF products, and compares four forward/backward-error filters.

The matcher treats pixel value `0` as missing. It returns horizontal displacement `u` (columns, positive east/right) and vertical displacement `v` (rows, positive south/down), both in pixels. The value `-128` is nodata. Correlations are exhaustive over the search window, so the full raster with a 25-pixel block and 40-pixel radius is a substantial GPU job.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from numba import cuda
from osgeo import gdal

from block_matching import block_matching_masked_ncc_uint_nonzero_fb

gdal.UseExceptions()

DATA_DIR = Path.cwd()
REFERENCE = DATA_DIR / "Sentinel2_PAN_20180804.tif"
TARGET = DATA_DIR / "Sentinel2_PAN_20200823.tif"

if not cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU and working Numba CUDA driver are required.")

print("CUDA device:", cuda.get_current_device().name)

## GeoTIFF writer

Every output copies the reference raster's transform and projection. Integer products use predictor 2 and floating-point products use predictor 3.

In [ ]:
def writetiff(filename, data, reference=REFERENCE, nodata=None, compress=True):
    with gdal.Open(str(reference), gdal.GA_ReadOnly) as src:
        if data.ndim != 2:
            raise ValueError("data must be a 2-D array")

        expected_shape = (src.RasterYSize, src.RasterXSize)
        if data.shape != expected_shape:
            raise ValueError(
                f"data shape {data.shape} does not match reference {expected_shape}"
            )

        dtype_map = {
            np.dtype("uint8"): gdal.GDT_Byte,
            np.dtype("int8"): gdal.GDT_Int8,
            np.dtype("uint16"): gdal.GDT_UInt16,
            np.dtype("int16"): gdal.GDT_Int16,
            np.dtype("uint32"): gdal.GDT_UInt32,
            np.dtype("int32"): gdal.GDT_Int32,
            np.dtype("float32"): gdal.GDT_Float32,
            np.dtype("float64"): gdal.GDT_Float64,
        }
        dtype = np.dtype(data.dtype)
        if dtype not in dtype_map:
            raise TypeError(f"Unsupported dtype: {dtype}")

        options = ["TILED=YES"]
        if compress:
            predictor = "PREDICTOR=2" if np.issubdtype(dtype, np.integer) else "PREDICTOR=3"
            options += ["COMPRESS=DEFLATE", predictor]

        dst = gdal.GetDriverByName("GTiff").Create(
            str(filename), data.shape[1], data.shape[0], 1, dtype_map[dtype], options=options
        )
        if dst is None:
            raise RuntimeError(f"Could not create output raster: {filename}")

        try:
            dst.SetGeoTransform(src.GetGeoTransform())
            dst.SetProjection(src.GetProjection())
            band = dst.GetRasterBand(1)
            if nodata is not None:
                band.SetNoDataValue(float(nodata))
            band.WriteArray(data)
            band.FlushCache()
            dst.FlushCache()
        finally:
            dst = None

## Read and validate the images

The CUDA function accepts `uint16`. Non-finite source values become zero, which is the matcher's missing-data convention. The range check prevents negative or greater-than-65535 values from silently wrapping during conversion. The rasters must already have identical grids; this notebook checks shape, geotransform, and projection but does not resample them.

In [ ]:
def read_uint16(path):
    with gdal.Open(str(path), gdal.GA_ReadOnly) as ds:
        array = ds.ReadAsArray()
        metadata = (array.shape, ds.GetGeoTransform(), ds.GetProjection())

    finite = np.isfinite(array)
    if finite.any() and (array[finite].min() < 0 or array[finite].max() > 65535):
        raise ValueError(f"Finite values in {path.name} do not fit in uint16")
    return np.where(finite, array, 0).astype(np.uint16), metadata


p, p_metadata = read_uint16(REFERENCE)
q, q_metadata = read_uint16(TARGET)

if p_metadata != q_metadata:
    raise ValueError("The input rasters are not on the same grid and CRS")

print("shape:", p.shape)
print("2018 nonzero pixels:", np.count_nonzero(p))
print("2020 nonzero pixels:", np.count_nonzero(q))

## Run block matching

A zero-valued mask processes every eligible center. Set mask entries to a nonzero value to omit clouds, water, unstable terrain, or any other unwanted locations. Borders wide enough to contain the block and full search window are omitted automatically. `min_valid_frac=0.5` (the default) requires at least half of the paired block pixels to be nonzero.

In [ ]:
block_size = 25       # must be positive and odd
search_radius = 40    # candidate offsets are -40 ... +40 pixels
mask = np.zeros(p.shape, dtype=np.uint16)

u, v, cmax8, cmean8, cstd8, sstd, tstd, efb2 = (
    block_matching_masked_ncc_uint_nonzero_fb(
        p, q, mask, block_size, search_radius, nthreads_exp=8
    )
)

print("valid forward matches:", np.count_nonzero((u != -128) & (v != -128)))

## Derive and write displacement products

`rho` is displacement magnitude in pixels and `phi` is direction in radians from the positive column axis, increasing toward positive rows. `efb` is the Euclidean closure error: the norm of the forward displacement plus its reverse-match displacement. A small value is more consistent. The implementation uses `65535` in `efb2` when no reverse match exists; this notebook represents that as NaN.

In [ ]:
uf = u.astype(np.float32)
vf = v.astype(np.float32)
match_invalid = (u == -128) | (v == -128)
backward_unavailable = efb2 == np.iinfo(np.uint16).max

rho = np.hypot(uf, vf).astype(np.float32)
phi = np.arctan2(vf, uf).astype(np.float32)
rho[match_invalid] = np.nan
phi[match_invalid] = np.nan

efb = np.sqrt(efb2.astype(np.float32))
efb[match_invalid | backward_unavailable] = np.nan

writetiff(DATA_DIR / "Sentinel2_u.tif", u, nodata=-128)
writetiff(DATA_DIR / "Sentinel2_v.tif", v, nodata=-128)
writetiff(DATA_DIR / "Sentinel2_rho.tif", rho, nodata=np.nan)
writetiff(DATA_DIR / "Sentinel2_phi.tif", phi, nodata=np.nan)
writetiff(DATA_DIR / "Sentinel2_efb.tif", efb, nodata=np.nan)

The matcher also returns three NCC diagnostics scaled from 0–1 to 0–255 (`cmax8`, `cmean8`, and `cstd8`) and the standard deviations of the winning source and target blocks in the original intensity units (`sstd` and `tstd`). They remain in memory here and can be written with the same helper if needed.

## Filter by forward/backward consistency

Reload the two relevant products, retain matches whose closure error is at most each threshold, and write four filtered magnitude rasters. Non-finite closure errors are rejected explicitly.

In [ ]:
with gdal.Open(str(DATA_DIR / "Sentinel2_rho.tif"), gdal.GA_ReadOnly) as ds:
    rho_from_disk = ds.ReadAsArray()

with gdal.Open(str(DATA_DIR / "Sentinel2_efb.tif"), gdal.GA_ReadOnly) as ds:
    efb_from_disk = ds.ReadAsArray()

thresholds = [0, 1, np.sqrt(2), 2]
rho_filtered = {}

for threshold in thresholds:
    rho_fb = rho_from_disk.copy()
    reject = ~np.isfinite(efb_from_disk) | (efb_from_disk > threshold)
    rho_fb[reject] = np.nan
    rho_filtered[threshold] = rho_fb
    writetiff(
        DATA_DIR / f"Sentinel2_rho_efb{threshold:.2f}.tif",
        rho_fb,
        nodata=np.nan,
    )

## Compare the filtered displacement magnitudes

All panels share one color scale. The 99th percentile limits the influence of a small number of large displacements; change `vmax` to `search_radius * sqrt(2)` to display the complete possible magnitude range.

In [ ]:
finite_rho = rho_from_disk[np.isfinite(rho_from_disk)]
vmax = np.nanpercentile(finite_rho, 99) if finite_rho.size else 1.0

fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True)
last_image = None
for ax, threshold in zip(axes.flat, thresholds):
    image = ax.imshow(
        rho_filtered[threshold], cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
    )
    last_image = image
    retained = np.count_nonzero(np.isfinite(rho_filtered[threshold]))
    ax.set_title(f"EFB ≤ {threshold:.2f} px ({retained:,} pixels)")
    ax.set_xlabel("column")
    ax.set_ylabel("row")

fig.colorbar(last_image, ax=axes, label="displacement magnitude (pixels)", shrink=0.9)
plt.show()